## Simple RAG Workflow
### A Retrieval-Augmented Generation (RAG) system combines a text generation model with an external knowledge store.

### Document collection

### Start with a corpus of documents, web pages, or other text sources.
### Split text into smaller chunks if needed.
### Embedding and indexing

###### Simple RAG Workflow
A Retrieval-Augmented Generation (RAG) system combines a text generation model with an external knowledge store.

### Document collection
- Start with a corpus of documents, web pages, or other text sources.
- Split text into smaller chunks if needed.

### Embedding and indexing
- Convert each text chunk into embeddings using a vector encoder.
- Store embeddings in a vector index for fast similarity search.

### Query processing
- Embed the user query with the same encoder.
- Retrieve the most relevant chunks from the index using similarity search.

### Augmented generation
- Send retrieved chunks and the original question to the language model.
- Generate an answer grounded in the retrieved context.

### Benefits
- Reduces hallucinations by grounding answers in real documents.
- Gives the model access to information beyond its training data.
- Useful for question answering, chat, and knowledge-heavy tasks.Convert each text chunk into embeddings using a ve

In [26]:
from langchain_ollama.llms import OllamaLLM
from langchain_core.prompts import PromptTemplate
from langchain_ollama.embeddings import OllamaEmbeddings

from joblib import Parallel, delayed
import pandas as pd

llm = OllamaLLM(model="gemma4:12b", base_url="http://localhost:11434")
embeddings = OllamaEmbeddings(model="nomic-embed-text",base_url="http://localhost:11434")

llm.invoke("What is the capital of France?")

'The capital of France is **Paris**.'

In [27]:
embeddings.embed_query("What is the capital of France?")


[-0.001664166,
 0.02265449,
 -0.15312794,
 -0.020760717,
 0.055857286,
 0.03561412,
 -0.03564022,
 0.028330645,
 -0.003189111,
 -0.030222103,
 -0.049998514,
 -0.023360452,
 0.075036295,
 -0.074831426,
 0.010357425,
 -0.02293267,
 -0.060387842,
 0.018458739,
 0.023994658,
 0.030462045,
 -0.021749869,
 -0.020172365,
 -0.012428026,
 -0.01512371,
 0.12805846,
 0.013496664,
 0.03565816,
 0.0047961003,
 -0.059952296,
 -0.0071165105,
 -0.02978324,
 -0.0021988396,
 -0.025813688,
 -0.0014836831,
 0.06132012,
 -0.056180768,
 -0.008535014,
 0.0059998333,
 0.046691295,
 0.044920925,
 0.024217144,
 -0.03384191,
 -0.0072720004,
 0.026453758,
 0.06663054,
 0.04211755,
 -0.043770626,
 0.026120571,
 0.03343684,
 -0.003885163,
 -0.027370093,
 0.01511509,
 -0.035123877,
 -0.0074163843,
 0.01209763,
 0.006080969,
 0.035430647,
 0.06540446,
 0.062619485,
 -0.012340835,
 0.07316339,
 0.00713486,
 0.0021280919,
 0.05580879,
 0.033675093,
 -0.0056897695,
 -0.017036844,
 0.10917971,
 0.022679485,
 -0.04316031,

In [29]:
CORPUS = [
    "In the quiet village of Larkspur, where the houses leaned toward one another as if listening, an old clockmaker named Ansel lived above his workshop. The room was filled with pendulums and winding keys, with clocks of every shape and age ticking in soft, steady chorus. Ansel had learned the craft from his father, and every morning he opened the wooden shutters and watched the sunlight bounce across brass faces while the familiar rhythm of the workshop set the pace of his day. He had no family nearby, but the villagers often stopped by for repairs, for advice, or simply to enjoy the warmth of the room. They said his hands worked with a kind of gentle patience, the sort that understood how delicate time could be. Though he was content, there was a quiet longing inside him for a story he could not name, a sense that the world beyond Larkspur held secrets he had yet to discover.",
    "One autumn evening, as the leaves outside rustled like pages being turned, a young traveler arrived at Ansel's door. She carried a dusty lantern and a leather satchel worn soft by many journeys. Her eyes were the deep green of wet moss, and her boots were caked with earth from distant roads. In her hand she held a pocket watch wrapped carefully in cloth. It was unlike any timepiece Ansel had ever seen: its case was etched with floral patterns that seemed to shift when he glanced away, and a faint, blue light glowed from within. The traveler spoke quietly, explaining that the watch had been found in a forest temple far from Larkspur and that it had refused to count the hours since it had been made. She wanted Ansel to fix it, though she admitted she did not understand how ordinary mechanics could mend something that felt so extraordinary.",
    "As Ansel unwrapped the pocket watch and began to examine it, a chill ran through him that had nothing to do with the evening air. The glass face was cool under his fingers, and beneath it he saw no ordinary gears. Instead, there was a small glass orb suspended in the center, swirling with blue light and tiny silver motes that floated like stars in a bowl of midnight. When he touched the orb, a flicker of memory flashed behind his eyelids: moonlit paths, the hush of ancient trees, a hidden glade where wishes had once been made. It felt as though the watch contained a story folded into its heart, a history of travels and hopes. Ansel realized that this repair would not be like the others. He would have to listen as much as he would have to work, for the watch seemed to ask not just for oil and gears but for understanding.",
    "Over the following days, Ansel and the traveler walked through the village and shared tales by the light of the workshop lamp. She spoke of a mountain pass where stars seemed to fall to the ground, of markets under canopies of woven lantern light, and of a hidden grove where an old watchmaker had once placed a wish. Ansel told her about Larkspur's narrow lanes, the annual harvest feast, and the clocks that had measured generations of laughter and grief. The watch, meanwhile, sat on his workbench, humming softly as if it listened. When night fell and the village grew hushed, the traveler would take out the watch and whisper questions into it. The orb inside responded with visions that flickered like fireflies — a distant bell tower, a child running through dew, a door glowing in the depths of a forest. Ansel began to sense that the watch's glow was a bridge between what was lost and what could be found.",
    "On the morning she planned to leave, Ansel wound the watch one last time and watched the blue light grow steady. The traveler placed her hand over his and spoke of the hidden glade again, of wishes and the power of kind deeds. She said the watch had been made by someone who believed that time could remind people of the paths they had forgotten. As they stood at the door, the village bathed in soft sunlight, Ansel felt a change within himself. The old longing had been replaced by a gentle certainty: that stories and moments, like the gears of a clock, were meant to be shared and set in motion. The traveler walked away down the cobbled lane, the watch safe in her satchel, and Ansel watched until she disappeared beyond the bend. The clocks in his workshop kept their steady time, but now he understood that time was also the space where magic could be found in the ordinary, and that every visitor might carry a story waiting to be wound into his own.",
]
from langchain_core.documents import Document
docs = [Document(page_content=doc) for doc in CORPUS] #simple text splitting on periods for demonstration; in practice, use a more robust method


docs

[Document(metadata={}, page_content='In the quiet village of Larkspur, where the houses leaned toward one another as if listening, an old clockmaker named Ansel lived above his workshop. The room was filled with pendulums and winding keys, with clocks of every shape and age ticking in soft, steady chorus. Ansel had learned the craft from his father, and every morning he opened the wooden shutters and watched the sunlight bounce across brass faces while the familiar rhythm of the workshop set the pace of his day. He had no family nearby, but the villagers often stopped by for repairs, for advice, or simply to enjoy the warmth of the room. They said his hands worked with a kind of gentle patience, the sort that understood how delicate time could be. Though he was content, there was a quiet longing inside him for a story he could not name, a sense that the world beyond Larkspur held secrets he had yet to discover.'),
 Document(metadata={}, page_content="One autumn evening, as the leaves o

## Chunking Logic

### Fixed Size Chunking: 
Fixed-size chunking splits text into equal-length pieces, usually by characters, tokens, or words, without looking at sentence or paragraph boundaries.


#### Pros:

<li> Simple to implement
<li> Works consistently across documents
<li> Easy to index and search


#### Cons:

<li> Can cut sentences or ideas in the middle
<li> May split important context across chunks
<li> Less efficient than semantic or sentence-aware chunking for meaning preservation

In [ ]:
CHUNK_SIZE = 200  # Adjust based on your needs
CHUNK_OVERLAP = 50  # Adjust based on your needs

chunks = []
for doc in docs:
    content = doc.page_content
    start = 0
    chunks += [content[i:i+CHUNK_SIZE] for i in range(0,len(content), CHUNK_SIZE-CHUNK_OVERLAP)]


print(f"Total chunks created: {len(chunks)}")
print(f"First chunk: {chunks[0][50:]}")
print(f"Second chunk: {chunks[1][:60]}") # checking the overlap between the first and second chunk



##Create the embeddings for the chunks
columns = ['chunk_id','chunk', 'embedding']
chunked_df = pd.DataFrame(columns=columns)

def embed_chunk(chunk_id, chunk):
    embeddings = OllamaEmbeddings(model="nomic-embed-text",base_url="http://localhost:11434")
    embeded_chunk = embeddings.embed_query(chunk)
    return pd.DataFrame([[chunk_id, chunk, embeded_chunk]], columns=columns)
from tqdm import tqdm
chunked_df = Parallel(n_jobs=-1)(delayed(embed_chunk)(i, chunk) for i, chunk in tqdm(enumerate(chunks)))
chunked_df = pd.concat(chunked_df, ignore_index=True)

print(f"Total chunks embedded: {len(chunked_df)}")
display(chunked_df.head())

Total chunks created: 32
First chunk:  leaned toward one another as if listening, an old clockmaker named Ansel lived above his workshop. The room was filled with pendulums and winding key
Second chunk: The room was filled with pendulums and winding keys, with cl


32it [00:02, 15.07it/s]


Total chunks embedded: 32


,chunk_id,chunk,embedding
0,0,"In the quiet village of Larkspur, where the ho...","[0.015897587, 0.020423532, -0.2034084, -0.0878..."
1,1,The room was filled with pendulums and winding...,"[-0.016528223, 0.048648495, -0.19957863, -0.09..."
2,2,"om his father, and every morning he opened the...","[-0.02556008, 0.016853891, -0.16262935, -0.049..."
3,3,kshop set the pace of his day. He had no famil...,"[-0.05213929, 0.032008253, -0.18695003, -0.038..."
4,4,he room. They said his hands worked with a kin...,"[0.043759968, 0.06697529, -0.17153932, -0.0906..."


### Retrieval 

In [52]:
import numpy as np
def calculate_similarity(query_embedding,chunk_embedding):
    """Calculate cosine similarity between two embeddings."""
    return np.dot(query_embedding, chunk_embedding) / (np.linalg.norm(query_embedding) * np.linalg.norm(chunk_embedding))


def retrieve_relevant_chunks(query_embedding: np.ndarray, 
                             top_k: int
                             ):
    
    """Retrieve the top_k most relevant chunks based on cosine similarity."""
    chunked_df['similarity'] = None
    results = Parallel(n_jobs=1
                       )(delayed(calculate_similarity)(query_embedding, chunk_embedding) for chunk_embedding in tqdm(chunked_df['embedding']))
    chunked_df['similarity'] = results
    chunked_df_sorted = chunked_df.sort_values(by='similarity', ascending=False)
    return chunked_df_sorted['chunk'].head(top_k).tolist()


TOP_K = 3
QUERY = "What village is being talked about in the story?"

RETRIEVED_CHUNKS = retrieve_relevant_chunks(embeddings.embed_query(QUERY), TOP_K)
print("Result for the query:\n", "\n".join(RETRIEVED_CHUNKS))


100%|██████████| 32/32 [00:00<00:00, 8025.46it/s]

Result for the query:
 Over the following days, Ansel and the traveler walked through the village and shared tales by the light of the workshop lamp. She spoke of a mountain pass where stars seemed to fall to the ground, of
In the quiet village of Larkspur, where the houses leaned toward one another as if listening, an old clockmaker named Ansel lived above his workshop. The room was filled with pendulums and winding key
remind people of the paths they had forgotten. As they stood at the door, the village bathed in soft sunlight, Ansel felt a change within himself. The old longing had been replaced by a gentle certain


In [54]:
### Evaluation of the Retrieval-Augmented Generation (RAG) System

system_prompt = f"""
You are a helpful assistant that judges if the documents retrieved from the previous step are 
relevant to the query. You will be given a query and a list of documents.
Your task is to evaluate each document and determine if it is relevant to the query.

Query : {QUERY}

Retrieved Chunks : {RETRIEVED_CHUNKS}
""" + """
Output format:
```
json
{
    {
      "chunk": "The content of the chunk",
      "is_relevant": true/false
    },
    ...
Relevance is determined by whether the chunk contains information that directly answers or relates to the query.  
}
```
"""

llm.invoke(system_prompt)



'```json\n{\n  "chunks": [\n    {\n      "chunk": "Over the following days, Ansel and the traveler walked through the village and shared tales by the light of the workshop lamp. She spoke of a mountain pass where stars seemed to fall to the ground, of",\n      "is_relevant": true\n    },\n    {\n      "chunk": "In the quiet village of Larkspur, where the houses leaned toward one another as if listening, an old clockmaker named Ansel lived above his workshop. The room was filled with pendulums and winding key",\n      "is_relevant": true\n    },\n    {\n      "chunk": "remind people of the paths they had forgotten. As they stood at the door, the village bathed in soft sunlight, Ansel felt a change within himself. The old longing had been replaced by a gentle certain",\n      "is_relevant": true\n    }\n  ]\n}\n```'

In [59]:

print('\n{\n  "chunks": [\n    {\n      "chunk": "Over the following days, Ansel and the traveler walked through the village and shared tales by the light of the workshop lamp. She spoke of a mountain pass where stars seemed to fall to the ground, of",\n      "is_relevant": true\n    },\n    {\n      "chunk": "In the quiet village of Larkspur, where the houses leaned toward one another as if listening, an old clockmaker named Ansel lived above his workshop. The room was filled with pendulums and winding key",\n      "is_relevant": true\n    },\n    {\n      "chunk": "remind people of the paths they had forgotten. As they stood at the door, the village bathed in soft sunlight, Ansel felt a change within himself. The old longing had been replaced by a gentle certain",\n      "is_relevant": true\n    }\n  ]\n}\n')


{
  "chunks": [
    {
      "chunk": "Over the following days, Ansel and the traveler walked through the village and shared tales by the light of the workshop lamp. She spoke of a mountain pass where stars seemed to fall to the ground, of",
      "is_relevant": true
    },
    {
      "chunk": "In the quiet village of Larkspur, where the houses leaned toward one another as if listening, an old clockmaker named Ansel lived above his workshop. The room was filled with pendulums and winding key",
      "is_relevant": true
    },
    {
      "chunk": "remind people of the paths they had forgotten. As they stood at the door, the village bathed in soft sunlight, Ansel felt a change within himself. The old longing had been replaced by a gentle certain",
      "is_relevant": true
    }
  ]
}



In [ ]:
## The output indicates that all three retrieved chunks are relevant to the query about the village being discussed in the story.
## Each chunk contains information that directly relates to the query, confirming their relevance.

## The above is just a demonstration of how to implement a Retrieval-Augmented Generation (RAG) system using the Ollama LLM and embeddings. The system retrieves relevant chunks of text based on a query and evaluates their relevance, providing a structured output in JSON format.